# HIV Surveillance Data Quality Assurance

This notebook performs comprehensive QA checks on HIV surveillance data.

**Includes:**
- Original QA checks (MSM proportion, sex/transmission mismatch, county divergence)
- Additional priority checks (date logic, lab values, missing data, duplicates)
- Export to Excel with separate sheets for each check

**Output:** `qa_results.xlsx`

In [1]:
# Import required libraries
import pandas as pd
import numpy as np
from datetime import datetime

print("Libraries loaded successfully")

Libraries loaded successfully


In [2]:
# Load the data
df = pd.read_csv(
    "simulated_data.csv",
    parse_dates=[
        "dob",
        "hiv_diagnosis_date",
        "aids_diagnosis_date",
        "last_lab_date",
        "report_date"
    ]
)

print(f"Total records loaded: {len(df):,}")
print(f"Date range: {df['hiv_diagnosis_date'].min()} to {df['hiv_diagnosis_date'].max()}")
df.head()

Total records loaded: 5,600
Date range: 2006-02-13 00:00:00 to 2026-02-12 00:00:00


,case_id,first_name,last_name,dob,sex_at_birth,current_gender,race_ethnicity,county,hiv_diagnosis_date,aids_diagnosis_date,transmission_category,last_cd4,last_viral_load,last_lab_date,report_date,record_status
0,1223,Paul,Hill,1932-06-29,Male,Male,White,St. Louis County,2009-06-29,NaT,MSM,1483.0,63098,2011-05-09,2009-11-04,Probable
1,3631,Mark,Howell,1984-12-25,Male,Female,Black,St. Louis City,2007-12-25,NaT,IDU,255.0,996335,2010-12-02,2008-12-02,Probable
2,12772,Rachel,Pop,1984-02-11,Male,Male,Black,Greene,2015-02-10,NaT,MSM,186.0,538627,2016-08-04,2015-11-29,Duplicate
3,384,Carlos,Stanton,1976-09-19,Female,Male,Black,Boone,2021-09-19,2022-12-24,IDU,1309.0,0,2021-10-26,2022-07-20,Confirmed
4,432,Kristin,West,1964-07-25,Male,Female,White,Boone,2012-07-25,NaT,MSM,299.0,979286,2012-10-09,2012-09-05,Probable


In [3]:
# Define reference groups and valid values
large_counties = ["St. Louis City", "St. Louis County", "Jackson"]
transmission_categories = ["MSM", "IDU", "Heterosexual", "MSM/IDU", "Perinatal", "Unknown"]
valid_sex = ["Male", "Female"]
valid_status = ["Confirmed", "Probable", "Duplicate"]
required_fields = ["case_id", "sex_at_birth", "hiv_diagnosis_date", "county", "record_status"]

today = pd.Timestamp.now().normalize()

# Dictionary to hold all QA results
qa_results = {}

print("Configuration loaded")

Configuration loaded


## Original QA Checks

In [4]:
# QA CHECK 1: High MSM Proportion in Small Counties
print("QA CHECK 1: High MSM proportion in small counties")

county_tx = (
    df.groupby(["county", "transmission_category"])
      .size()
      .unstack(fill_value=0)
)

county_tx["MSM_pct"] = county_tx["MSM"] / county_tx.sum(axis=1)

qa_msm_small_county = county_tx[
    (~county_tx.index.isin(large_counties)) &
    (county_tx["MSM_pct"] > 0.60)
].reset_index()

qa_msm_small_county["qa_flag"] = "High MSM proportion in small county"
qa_msm_small_county["check_type"] = "Geographic Pattern"

qa_results["01_MSM_Small_Counties"] = qa_msm_small_county
print(f"→ {len(qa_msm_small_county)} counties flagged")
qa_msm_small_county

QA CHECK 1: High MSM proportion in small counties
→ 6 counties flagged


transmission_category,county,Heterosexual,IDU,MSM,MSM/IDU,Perinatal,Unknown,MSM_pct,qa_flag,check_type
0,Audrain,1,0,12,2,0,0,0.800000,High MSM proportion in small county,Geographic Pattern
1,Christian,1,2,9,2,0,0,0.642857,High MSM proportion in small county,Geographic Pattern
2,Moniteau,3,1,10,1,1,0,0.625000,High MSM proportion in small county,Geographic Pattern
3,Osage,5,2,13,0,0,0,0.650000,High MSM proportion in small county,Geographic Pattern
4,Pettis,5,0,13,0,0,0,0.722222,High MSM proportion in small county,Geographic Pattern
5,Pulaski,2,1,8,1,1,0,0.615385,High MSM proportion in small county,Geographic Pattern


In [5]:
# QA CHECK 2: Female Sex at Birth with MSM Transmission
print("QA CHECK 2: Female sex at birth with MSM transmission")

qa_female_msm = df[
    (df["sex_at_birth"] == "Female") &
    (df["transmission_category"] == "MSM")
].copy()

qa_female_msm["qa_flag"] = "Female sex at birth with MSM transmission"
qa_female_msm["check_type"] = "Sex/Transmission Mismatch"

qa_results["02_Female_MSM"] = qa_female_msm
print(f"→ {len(qa_female_msm)} records flagged")
qa_female_msm.head(10)

QA CHECK 2: Female sex at birth with MSM transmission
→ 49 records flagged


,case_id,first_name,last_name,dob,sex_at_birth,current_gender,race_ethnicity,county,hiv_diagnosis_date,aids_diagnosis_date,transmission_category,last_cd4,last_viral_load,last_lab_date,report_date,record_status,qa_flag,check_type
44,11238,Joseph,Patterso,1973-07-27,Female,Male,Black,Cass,2013-07-27,NaT,MSM,528.0,0,2014-05-04,2014-05-20,Duplicate,Female sex at birth with MSM transmission,Sex/Transmission Mismatch
220,2752,Jonathan,Smith,2002-11-01,Female,Male,Black,St. Louis County,2025-10-31,NaT,MSM,514.0,389475,2026-07-23,2025-12-02,Probable,Female sex at birth with MSM transmission,Sex/Transmission Mismatch
487,3673,William,Wiggins,1980-12-02,Female,Male,Black,Greene,2020-12-02,NaT,MSM,1343.0,-5,2023-01-27,2021-04-01,Probable,Female sex at birth with MSM transmission,Sex/Transmission Mismatch
521,2858,Nicholas,Parker,1942-05-04,Female,Female,Hispanic,St. Louis County,2019-05-04,2021-06-07,MSM,214.0,640839,2019-06-16,2019-10-03,Confirmed,Female sex at birth with MSM transmission,Sex/Transmission Mismatch
631,3222,Juan,Ferguson,2000-08-03,Female,Male,White,Jackson,2024-08-03,2027-06-04,MSM,1496.0,557109,2026-08-05,2024-08-25,Confirmed,Female sex at birth with MSM transmission,Sex/Transmission Mismatch
732,14221,Jason,Hammon,1979-12-13,Female,Male,Hispanic,Christian,2009-12-12,NaT,MSM,-10.0,372283,2010-03-14,2010-08-12,Duplicate,Female sex at birth with MSM transmission,Sex/Transmission Mismatch
820,4912,Travis,Frazier,1939-04-29,Female,Male,Black,Clay,2011-04-29,2015-03-14,MSM,415.0,388105,2011-07-19,2011-06-22,Confirmed,Female sex at birth with MSM transmission,Sex/Transmission Mismatch
920,14,Sharon,Johns,1933-05-29,Female,Male,Black,Greene,2010-05-29,2009-10-03,MSM,1032.0,-5,2012-11-08,2011-03-06,Probable,Female sex at birth with MSM transmission,Sex/Transmission Mismatch
950,4221,Jason,Hammond,1979-12-13,Female,Male,Hispanic,Jackson,2009-12-12,NaT,MSM,-10.0,372283,2010-03-14,2010-08-12,Confirmed,Female sex at birth with MSM transmission,Sex/Transmission Mismatch
1065,14778,Theresa,Warre,1972-06-04,Female,Male,Black,Boone,2016-06-04,2019-03-04,MSM,NaN,271191,2017-01-24,2017-01-21,Duplicate,Female sex at birth with MSM transmission,Sex/Transmission Mismatch


In [6]:
# QA CHECK 3: Counties with Unusual Transmission Distributions
print("QA CHECK 3: Counties with unusual transmission distributions")

# Calculate state-level transmission distribution
state_tx_dist = df["transmission_category"].value_counts(normalize=True)

# Get county-level proportions
county_tx_full = county_tx[transmission_categories]
county_tx_prop = county_tx_full.div(county_tx_full.sum(axis=1), axis=0)

# Calculate divergence from state pattern
def divergence_from_state(row):
    return np.sum(
        np.abs(row - state_tx_dist.reindex(row.index, fill_value=0))
    )

divergence_scores = county_tx_prop.apply(divergence_from_state, axis=1)
threshold = divergence_scores.quantile(0.85)

qa_divergent_counties = divergence_scores[
    divergence_scores > threshold
].reset_index()

qa_divergent_counties.columns = ["county", "divergence_score"]
qa_divergent_counties["qa_flag"] = "Unusual transmission distribution"
qa_divergent_counties["check_type"] = "Geographic Pattern"

# Merge with county transmission data for context
qa_divergent_counties = qa_divergent_counties.merge(
    county_tx.reset_index(),
    on="county",
    how="left"
)

qa_results["03_Divergent_Counties"] = qa_divergent_counties
print(f"→ {len(qa_divergent_counties)} counties flagged")
qa_divergent_counties

QA CHECK 3: Counties with unusual transmission distributions
→ 5 counties flagged


,county,divergence_score,qa_flag,check_type,Heterosexual,IDU,MSM,MSM/IDU,Perinatal,Unknown,MSM_pct
0,Audrain,0.707865,Unusual transmission distribution,Geographic Pattern,1,0,12,2,0,0,0.800000
1,Callaway,0.329089,Unusual transmission distribution,Geographic Pattern,5,4,8,0,0,1,0.444444
2,Christian,0.432424,Unusual transmission distribution,Geographic Pattern,1,2,9,2,0,0,0.642857
3,Howard,0.482653,Unusual transmission distribution,Geographic Pattern,5,6,6,1,1,0,0.315789
4,Pettis,0.441573,Unusual transmission distribution,Geographic Pattern,5,0,13,0,0,0,0.722222


## New QA Checks - Date Logic

In [7]:
# QA CHECK 4: AIDS Diagnosis Before HIV Diagnosis
print("QA CHECK 4: AIDS diagnosis before HIV diagnosis")

qa_aids_before_hiv = df[
    (df["aids_diagnosis_date"].notna()) & 
    (df["hiv_diagnosis_date"].notna()) &
    (df["aids_diagnosis_date"] < df["hiv_diagnosis_date"])
].copy()

qa_aids_before_hiv["qa_flag"] = "AIDS diagnosis date before HIV diagnosis date"
qa_aids_before_hiv["check_type"] = "Date Logic Error"
qa_aids_before_hiv["days_difference"] = (
    qa_aids_before_hiv["hiv_diagnosis_date"] - qa_aids_before_hiv["aids_diagnosis_date"]
).dt.days

qa_results["04_AIDS_Before_HIV"] = qa_aids_before_hiv
print(f"→ {len(qa_aids_before_hiv)} records flagged")
qa_aids_before_hiv[["case_id", "hiv_diagnosis_date", "aids_diagnosis_date", "days_difference"]].head(10)

QA CHECK 4: AIDS diagnosis before HIV diagnosis
→ 117 records flagged


,case_id,hiv_diagnosis_date,aids_diagnosis_date,days_difference
27,2502,2011-10-17,2010-11-30,321
34,12547,2014-10-09,2013-11-28,315
158,2354,2020-06-08,2019-11-21,200
198,2936,2023-09-04,2023-07-11,55
309,4645,2015-11-09,2015-10-08,32
382,661,2024-07-04,2024-05-05,60
514,179,2015-09-04,2015-03-26,162
582,1784,2014-01-03,2013-05-11,237
616,2241,2018-11-07,2017-12-26,316
656,4099,2008-07-08,2008-01-04,186


In [8]:
# QA CHECK 5: Future Diagnosis Dates
print("QA CHECK 5: Future diagnosis dates")

qa_future_hiv = df[
    (df["hiv_diagnosis_date"].notna()) &
    (df["hiv_diagnosis_date"] > today)
].copy()
qa_future_hiv["qa_flag"] = "HIV diagnosis date in the future"
qa_future_hiv["check_type"] = "Date Logic Error"

qa_future_aids = df[
    (df["aids_diagnosis_date"].notna()) &
    (df["aids_diagnosis_date"] > today)
].copy()
qa_future_aids["qa_flag"] = "AIDS diagnosis date in the future"
qa_future_aids["check_type"] = "Date Logic Error"

qa_future_dates = pd.concat([qa_future_hiv, qa_future_aids], ignore_index=True)
qa_results["05_Future_Dates"] = qa_future_dates
print(f"→ {len(qa_future_dates)} records flagged")
qa_future_dates[["case_id", "hiv_diagnosis_date", "aids_diagnosis_date", "qa_flag"]].head(10)

QA CHECK 5: Future diagnosis dates
→ 324 records flagged


,case_id,hiv_diagnosis_date,aids_diagnosis_date,qa_flag
0,4054,2024-01-06,2028-05-22,AIDS diagnosis date in the future
1,4801,2022-04-22,2027-05-21,AIDS diagnosis date in the future
2,1633,2023-08-03,2028-09-05,AIDS diagnosis date in the future
3,4412,2025-11-28,2029-01-02,AIDS diagnosis date in the future
4,12228,2024-08-06,2028-07-02,AIDS diagnosis date in the future
5,3578,2025-12-26,2029-12-01,AIDS diagnosis date in the future
6,4051,2024-07-02,2028-01-24,AIDS diagnosis date in the future
7,3380,2025-09-02,2029-01-20,AIDS diagnosis date in the future
8,243,2022-11-07,2027-08-27,AIDS diagnosis date in the future
9,1770,2023-06-10,2027-07-24,AIDS diagnosis date in the future


In [9]:
# QA CHECK 6: Future Lab and Report Dates
print("QA CHECK 6: Future lab and report dates")

qa_future_lab = df[
    (df["last_lab_date"].notna()) &
    (df["last_lab_date"] > today)
].copy()
qa_future_lab["qa_flag"] = "Lab date in the future"
qa_future_lab["check_type"] = "Date Logic Error"

qa_future_report = df[
    (df["report_date"].notna()) &
    (df["report_date"] > today)
].copy()
qa_future_report["qa_flag"] = "Report date in the future"
qa_future_report["check_type"] = "Date Logic Error"

qa_future_other = pd.concat([qa_future_lab, qa_future_report], ignore_index=True)
qa_results["06_Future_Lab_Report"] = qa_future_other
print(f"→ {len(qa_future_other)} records flagged")
qa_future_other[["case_id", "last_lab_date", "report_date", "qa_flag"]].head(10)

QA CHECK 6: Future lab and report dates
→ 632 records flagged


,case_id,last_lab_date,report_date,qa_flag
0,2729,2026-08-30,2024-07-19,Lab date in the future
1,3972,2027-01-17,2024-03-18,Lab date in the future
2,4412,2027-08-23,2026-01-20,Lab date in the future
3,3800,2027-01-04,2024-10-07,Lab date in the future
4,4805,2026-08-22,2019-08-30,Lab date in the future
5,3172,2028-04-24,2025-08-13,Lab date in the future
6,102,2028-06-30,2026-11-23,Lab date in the future
7,3578,2027-07-14,2025-12-29,Lab date in the future
8,4738,2026-10-06,2025-11-14,Lab date in the future
9,3380,2028-06-06,2025-11-16,Lab date in the future


## New QA Checks - Lab Values

In [10]:
# QA CHECK 7: Negative Lab Values
print("QA CHECK 7: Negative lab values")

qa_negative_cd4 = df[
    (df["last_cd4"].notna()) &
    (df["last_cd4"] < 0)
].copy()
qa_negative_cd4["qa_flag"] = "Negative CD4 count"
qa_negative_cd4["check_type"] = "Lab Value Error"

qa_negative_vl = df[
    (df["last_viral_load"].notna()) &
    (df["last_viral_load"] < 0)
].copy()
qa_negative_vl["qa_flag"] = "Negative viral load"
qa_negative_vl["check_type"] = "Lab Value Error"

qa_negative_labs = pd.concat([qa_negative_cd4, qa_negative_vl], ignore_index=True)
qa_results["07_Negative_Lab_Values"] = qa_negative_labs
print(f"→ {len(qa_negative_labs)} records flagged")
print(f"CD4 negative: {len(qa_negative_cd4)}, VL negative: {len(qa_negative_vl)}")
qa_negative_labs[["case_id", "last_cd4", "last_viral_load", "qa_flag"]].head(10)

QA CHECK 7: Negative lab values
→ 608 records flagged
CD4 negative: 304, VL negative: 304


,case_id,last_cd4,last_viral_load,qa_flag
0,3972,-10.0,34798,Negative CD4 count
1,2502,-10.0,0,Negative CD4 count
2,3472,-10.0,741837,Negative CD4 count
3,1947,-10.0,716870,Negative CD4 count
4,163,-10.0,14925,Negative CD4 count
5,1354,-10.0,0,Negative CD4 count
6,3612,-10.0,0,Negative CD4 count
7,1662,-10.0,850430,Negative CD4 count
8,2862,-10.0,557868,Negative CD4 count
9,14585,-10.0,451239,Negative CD4 count


In [11]:
# QA CHECK 8: Extremely High Lab Values
print("QA CHECK 8: Extremely high lab values")

qa_high_cd4 = df[
    (df["last_cd4"].notna()) &
    (df["last_cd4"] > 2000)
].copy()
qa_high_cd4["qa_flag"] = "CD4 count > 2000 (unusually high)"
qa_high_cd4["check_type"] = "Lab Value Warning"

qa_high_vl = df[
    (df["last_viral_load"].notna()) &
    (df["last_viral_load"] > 1000000)
].copy()
qa_high_vl["qa_flag"] = "Viral load > 1,000,000 (extremely high)"
qa_high_vl["check_type"] = "Lab Value Warning"

qa_high_labs = pd.concat([qa_high_cd4, qa_high_vl], ignore_index=True)
qa_results["08_High_Lab_Values"] = qa_high_labs
print(f"→ {len(qa_high_labs)} records flagged")
if len(qa_high_labs) > 0:
    qa_high_labs[["case_id", "last_cd4", "last_viral_load", "qa_flag"]].head(10)
else:
    print("No records with extremely high lab values")

QA CHECK 8: Extremely high lab values
→ 0 records flagged
No records with extremely high lab values


## New QA Checks - Date Sequences

In [12]:
# QA CHECK 9: Report and Lab Dates Before Diagnosis
print("QA CHECK 9: Report/lab dates before diagnosis")

qa_report_before_dx = df[
    (df["report_date"].notna()) &
    (df["hiv_diagnosis_date"].notna()) &
    (df["report_date"] < df["hiv_diagnosis_date"])
].copy()
qa_report_before_dx["qa_flag"] = "Report date before HIV diagnosis date"
qa_report_before_dx["check_type"] = "Date Sequence Error"
qa_report_before_dx["days_difference"] = (
    qa_report_before_dx["hiv_diagnosis_date"] - qa_report_before_dx["report_date"]
).dt.days

qa_lab_before_dx = df[
    (df["last_lab_date"].notna()) &
    (df["hiv_diagnosis_date"].notna()) &
    (df["last_lab_date"] < df["hiv_diagnosis_date"])
].copy()
qa_lab_before_dx["qa_flag"] = "Lab date before HIV diagnosis date"
qa_lab_before_dx["check_type"] = "Date Sequence Warning"
qa_lab_before_dx["days_difference"] = (
    qa_lab_before_dx["hiv_diagnosis_date"] - qa_lab_before_dx["last_lab_date"]
).dt.days

qa_dates_before = pd.concat([qa_report_before_dx, qa_lab_before_dx], ignore_index=True)
qa_results["09_Dates_Before_Diagnosis"] = qa_dates_before
print(f"→ {len(qa_dates_before)} records flagged")
qa_dates_before[["case_id", "hiv_diagnosis_date", "report_date", "last_lab_date", "qa_flag"]].head(10)

QA CHECK 9: Report/lab dates before diagnosis
→ 157 records flagged


,case_id,hiv_diagnosis_date,report_date,last_lab_date,qa_flag
0,139,2015-09-15,2015-09-13,2017-07-13,Report date before HIV diagnosis date
1,1341,2009-12-10,2009-08-02,2010-03-08,Report date before HIV diagnosis date
2,146,2015-03-17,2014-11-28,2017-09-04,Report date before HIV diagnosis date
3,4755,2012-09-26,2012-08-14,2015-04-08,Report date before HIV diagnosis date
4,2114,2016-01-09,2015-09-15,2017-08-29,Report date before HIV diagnosis date
5,2259,2007-05-21,2007-03-03,2007-11-06,Report date before HIV diagnosis date
6,2679,2022-02-13,2021-09-05,2023-11-11,Report date before HIV diagnosis date
7,14585,2010-08-13,2010-04-29,2013-04-25,Report date before HIV diagnosis date
8,1497,2014-01-06,2013-09-17,2016-10-17,Report date before HIV diagnosis date
9,4160,2018-08-08,2018-02-28,2021-04-13,Report date before HIV diagnosis date


## New QA Checks - Age/Transmission Mismatches

In [13]:
# QA CHECK 10: Age at Diagnosis Issues
print("QA CHECK 10: Age at diagnosis issues")

df_temp = df.copy()
df_temp["age_at_hiv_dx"] = (
    (df_temp["hiv_diagnosis_date"] - df_temp["dob"]).dt.days / 365.25
)

adult_transmission = ["MSM", "Heterosexual", "IDU", "MSM/IDU"]

qa_pediatric_adult_tx = df_temp[
    (df_temp["age_at_hiv_dx"].notna()) &
    (df_temp["age_at_hiv_dx"] < 13) &
    (df_temp["transmission_category"].isin(adult_transmission))
].copy()
qa_pediatric_adult_tx["qa_flag"] = "Pediatric age (<13) with adult transmission category"
qa_pediatric_adult_tx["check_type"] = "Age/Transmission Mismatch"

qa_adult_perinatal = df_temp[
    (df_temp["age_at_hiv_dx"].notna()) &
    (df_temp["age_at_hiv_dx"] > 18) &
    (df_temp["transmission_category"] == "Perinatal")
].copy()
qa_adult_perinatal["qa_flag"] = "Perinatal transmission diagnosed at age > 18"
qa_adult_perinatal["check_type"] = "Age/Transmission Mismatch"

qa_age_issues = pd.concat([qa_pediatric_adult_tx, qa_adult_perinatal], ignore_index=True)
qa_results["10_Age_Diagnosis_Issues"] = qa_age_issues
print(f"→ {len(qa_age_issues)} records flagged")
qa_age_issues[["case_id", "dob", "hiv_diagnosis_date", "age_at_hiv_dx", "transmission_category", "qa_flag"]].head(10)

QA CHECK 10: Age at diagnosis issues
→ 157 records flagged


,case_id,dob,hiv_diagnosis_date,age_at_hiv_dx,transmission_category,qa_flag
0,3472,1959-06-14,2016-06-13,56.999316,Perinatal,Perinatal transmission diagnosed at age > 18
1,3123,1989-02-12,2015-02-12,25.998631,Perinatal,Perinatal transmission diagnosed at age > 18
2,2677,1991-10-20,2016-10-19,24.999316,Perinatal,Perinatal transmission diagnosed at age > 18
3,170,1944-04-03,2020-04-03,76.000000,Perinatal,Perinatal transmission diagnosed at age > 18
4,1477,1975-09-20,2015-09-20,40.000000,Perinatal,Perinatal transmission diagnosed at age > 18
5,12968,1952-07-09,2019-07-09,66.997947,Perinatal,Perinatal transmission diagnosed at age > 18
6,11246,1938-04-09,2008-04-08,69.998631,Perinatal,Perinatal transmission diagnosed at age > 18
7,4696,1981-04-16,2012-04-15,30.997947,Perinatal,Perinatal transmission diagnosed at age > 18
8,2245,1962-02-28,2008-02-28,45.998631,Perinatal,Perinatal transmission diagnosed at age > 18
9,13810,1985-09-18,2014-09-18,28.999316,Perinatal,Perinatal transmission diagnosed at age > 18


## New QA Checks - Missing Data

In [15]:
# QA CHECK 11: Missing Required Fields
print("QA CHECK 11: Missing required fields")

missing_data_summary = []
for field in required_fields:
    missing_count = df[field].isna().sum()
    missing_pct = missing_count / len(df) * 100
    missing_data_summary.append({
        "field_name": field,
        "missing_count": missing_count,
        "missing_percentage": missing_pct,
        "total_records": len(df)
    })

qa_missing_summary = pd.DataFrame(missing_data_summary)
qa_results["11_Missing_Required_Summary"] = qa_missing_summary

# Detailed records with missing required fields
qa_missing_required = pd.DataFrame()
for field in required_fields:
    temp = df[df[field].isna()].copy()
    if len(temp) > 0:
        temp["qa_flag"] = f"Missing required field: {field}"
        temp["missing_field"] = field
        temp["check_type"] = "Missing Required Data"
        qa_missing_required = pd.concat([qa_missing_required, temp], ignore_index=True)

qa_results["11_Missing_Required_Detail"] = qa_missing_required
print(f"→ {len(qa_missing_required)} records flagged")
qa_missing_summary

QA CHECK 11: Missing required fields
→ 0 records flagged


,field_name,missing_count,missing_percentage,total_records
0,case_id,0,0.0,5600
1,sex_at_birth,0,0.0,5600
2,hiv_diagnosis_date,0,0.0,5600
3,county,0,0.0,5600
4,record_status,0,0.0,5600


In [16]:
# QA CHECK 12: Missing Important Fields
print("QA CHECK 12: Missing important fields")

qa_missing_race = df[df["race_ethnicity"].isna()].copy()
qa_missing_race["qa_flag"] = "Missing race/ethnicity"
qa_missing_race["check_type"] = "Missing Important Data"

qa_missing_transmission = df[df["transmission_category"].isna()].copy()
qa_missing_transmission["qa_flag"] = "Missing transmission category"
qa_missing_transmission["check_type"] = "Missing Important Data"

qa_missing_important = pd.concat([qa_missing_race, qa_missing_transmission], ignore_index=True)
qa_results["12_Missing_Important_Fields"] = qa_missing_important
print(f"→ {len(qa_missing_important)} records flagged")
print(f"Missing race/ethnicity: {len(qa_missing_race)}")
print(f"Missing transmission: {len(qa_missing_transmission)}")

QA CHECK 12: Missing important fields
→ 501 records flagged
Missing race/ethnicity: 241
Missing transmission: 260


## New QA Checks - Data Integrity

In [17]:
# QA CHECK 13: Duplicate Case IDs
print("QA CHECK 13: Duplicate case IDs")

duplicate_case_ids = df[df.duplicated(subset=["case_id"], keep=False)].copy()
duplicate_case_ids = duplicate_case_ids.sort_values("case_id")
duplicate_case_ids["qa_flag"] = "Duplicate case ID"
duplicate_case_ids["check_type"] = "Data Integrity"

qa_results["13_Duplicate_Case_IDs"] = duplicate_case_ids
print(f"→ {duplicate_case_ids['case_id'].nunique()} unique case IDs duplicated")
print(f"→ {len(duplicate_case_ids)} total duplicate records")

if len(duplicate_case_ids) > 0:
    duplicate_case_ids[["case_id", "first_name", "last_name", "dob", "hiv_diagnosis_date"]].head(10)

QA CHECK 13: Duplicate case IDs
→ 0 unique case IDs duplicated
→ 0 total duplicate records


In [18]:
# QA CHECK 14: Records Marked as Duplicate Status
print("QA CHECK 14: Records marked as duplicate status")

qa_duplicate_status = df[df["record_status"] == "Duplicate"].copy()
qa_duplicate_status["qa_flag"] = "Record status marked as Duplicate"
qa_duplicate_status["check_type"] = "Data Integrity"

qa_results["14_Duplicate_Status"] = qa_duplicate_status
print(f"→ {len(qa_duplicate_status)} records flagged")
qa_duplicate_status[["case_id", "first_name", "last_name", "hiv_diagnosis_date", "record_status"]].head(10)

QA CHECK 14: Records marked as duplicate status
→ 600 records flagged


,case_id,first_name,last_name,hiv_diagnosis_date,record_status
2,12772,Rachel,Pop,2015-02-10,Duplicate
34,12547,Joshua,William,2014-10-09,Duplicate
42,11289,Heather,Brow,2010-02-17,Duplicate
44,11238,Joseph,Patterso,2013-07-27,Duplicate
69,13251,Stacey,Malon,2007-12-30,Duplicate
73,12228,James,Hane,2024-08-06,Duplicate
98,13349,Mary,Austi,2007-08-01,Duplicate
103,10532,Kathryn,Roja,2006-06-05,Duplicate
113,13300,Raymond,Moren,2008-05-23,Duplicate
125,11370,Robert,Duart,2019-09-24,Duplicate


In [19]:
# QA CHECK 15: Invalid Categorical Values
print("QA CHECK 15: Invalid categorical values")

qa_invalid_sex = df[
    (df["sex_at_birth"].notna()) &
    (~df["sex_at_birth"].isin(valid_sex))
].copy()
qa_invalid_sex["qa_flag"] = "Invalid sex at birth value"
qa_invalid_sex["check_type"] = "Invalid Category"

qa_invalid_transmission = df[
    (df["transmission_category"].notna()) &
    (~df["transmission_category"].isin(transmission_categories))
].copy()
qa_invalid_transmission["qa_flag"] = "Invalid transmission category"
qa_invalid_transmission["check_type"] = "Invalid Category"

qa_invalid_status = df[
    (df["record_status"].notna()) &
    (~df["record_status"].isin(valid_status))
].copy()
qa_invalid_status["qa_flag"] = "Invalid record status"
qa_invalid_status["check_type"] = "Invalid Category"

qa_invalid_categories = pd.concat([
    qa_invalid_sex, 
    qa_invalid_transmission, 
    qa_invalid_status
], ignore_index=True)

qa_results["15_Invalid_Categories"] = qa_invalid_categories
print(f"→ {len(qa_invalid_categories)} records flagged")

if len(qa_invalid_categories) > 0:
    qa_invalid_categories[["case_id", "sex_at_birth", "transmission_category", "record_status", "qa_flag"]].head(10)

QA CHECK 15: Invalid categorical values
→ 0 records flagged


## Generate Executive Summary

In [20]:
# Create Executive Summary
print("Generating executive summary...")

summary_data = []
for sheet_name, qa_df in qa_results.items():
    if len(qa_df) > 0:
        # Get unique check type if available
        check_type = qa_df["check_type"].iloc[0] if "check_type" in qa_df.columns else "N/A"
        
        # Count unique cases if case_id exists
        if "case_id" in qa_df.columns:
            unique_cases = qa_df["case_id"].nunique()
        else:
            unique_cases = len(qa_df)
        
        summary_data.append({
            "QA_Check": sheet_name.replace("_", " "),
            "Check_Type": check_type,
            "Records_Flagged": len(qa_df),
            "Unique_Cases": unique_cases,
            "Percent_of_Total": round(unique_cases / len(df) * 100, 2)
        })

summary_df = pd.DataFrame(summary_data)
summary_df = summary_df.sort_values("Records_Flagged", ascending=False)

# Add overall statistics
overall_stats = pd.DataFrame([{
    "Metric": "Total Records in Dataset",
    "Value": len(df)
}, {
    "Metric": "Total QA Flags Generated",
    "Value": sum(summary_df["Records_Flagged"])
}, {
    "Metric": "Unique Cases Flagged",
    "Value": len(df[df["case_id"].isin(
        pd.concat([qa_df for qa_df in qa_results.values() 
                  if "case_id" in qa_df.columns])["case_id"].unique()
    )])
}, {
    "Metric": "Percent of Cases Flagged",
    "Value": f"{len(df[df['case_id'].isin(pd.concat([qa_df for qa_df in qa_results.values() if 'case_id' in qa_df.columns])['case_id'].unique())]) / len(df) * 100:.2f}%"
}, {
    "Metric": "Analysis Date",
    "Value": datetime.now().strftime('%Y-%m-%d %H:%M:%S')
}])

qa_results = {"00_Executive_Summary": summary_df, "00_Overall_Stats": overall_stats, **qa_results}

print("Summary generated successfully")
summary_df

Generating executive summary...
Summary generated successfully


,QA_Check,Check_Type,Records_Flagged,Unique_Cases,Percent_of_Total
5,06 Future Lab Report,Date Logic Error,632,507,9.05
6,07 Negative Lab Values,Lab Value Error,608,594,10.61
11,14 Duplicate Status,Data Integrity,600,600,10.71
10,12 Missing Important Fields,Missing Important Data,501,495,8.84
4,05 Future Dates,Date Logic Error,324,324,5.79
7,09 Dates Before Diagnosis,Date Sequence Error,157,157,2.80
8,10 Age Diagnosis Issues,Age/Transmission Mismatch,157,157,2.80
3,04 AIDS Before HIV,Date Logic Error,117,117,2.09
1,02 Female MSM,Sex/Transmission Mismatch,49,49,0.88
0,01 MSM Small Counties,Geographic Pattern,6,6,0.11


In [21]:
# View overall statistics
overall_stats

,Metric,Value
0,Total Records in Dataset,5600
1,Total QA Flags Generated,3161
2,Unique Cases Flagged,2299
3,Percent of Cases Flagged,41.05%
4,Analysis Date,2026-02-13 15:32:32


## Export to Excel

In [22]:
# Export all results to Excel
print("Exporting results to Excel...")

output_file = "qa_results.xlsx"

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    for sheet_name, qa_df in qa_results.items():
        # Excel sheet names have max 31 characters
        clean_sheet_name = sheet_name[:31]
        qa_df.to_excel(writer, sheet_name=clean_sheet_name, index=False)
        
        # Auto-adjust column widths
        worksheet = writer.sheets[clean_sheet_name]
        for idx, col in enumerate(qa_df.columns):
            try:
                max_length = max(
                    qa_df[col].astype(str).str.len().max(),
                    len(str(col))
                ) + 2
                # Handle cases where column index > 26 (beyond 'Z')
                if idx < 26:
                    col_letter = chr(65 + idx)
                else:
                    col_letter = chr(64 + idx // 26) + chr(65 + idx % 26)
                worksheet.column_dimensions[col_letter].width = min(max_length, 50)
            except:
                pass  # Skip if column width adjustment fails

print(f"\n✓ QA results exported to: {output_file}")
print(f"✓ Total sheets created: {len(qa_results)}")

Exporting results to Excel...

✓ QA results exported to: qa_results.xlsx
✓ Total sheets created: 18


## Final Summary

In [23]:
# Print final summary
print("="*80)
print("QA ANALYSIS COMPLETE")
print("="*80)
print(f"\nTotal records analyzed: {len(df):,}")
print(f"Total QA flags: {sum(summary_df['Records_Flagged']):,}")
print(f"\nTop 5 QA Issues:")
for idx, row in summary_df.head(5).iterrows():
    print(f"  {idx+1}. {row['QA_Check']}: {row['Records_Flagged']:,} records ({row['Percent_of_Total']}%)")

print(f"\n✓ Review the Excel file '{output_file}' for detailed findings.")
print("="*80)

QA ANALYSIS COMPLETE

Total records analyzed: 5,600
Total QA flags: 3,161

Top 5 QA Issues:
  6. 06 Future Lab Report: 632 records (9.05%)
  7. 07 Negative Lab Values: 608 records (10.61%)
  12. 14 Duplicate Status: 600 records (10.71%)
  11. 12 Missing Important Fields: 501 records (8.84%)
  5. 05 Future Dates: 324 records (5.79%)

✓ Review the Excel file 'qa_results.xlsx' for detailed findings.
